# 02 — Data Enrichment
## India Tourist Crowd Forecasting System

### What this notebook does
Enriches the 12,655 places dataset with:
- **Real Google Popular Times** busyness data (fetched for 1000+ review places)
- **Real opening hours** from Google Place Details API
- **Real Google Trends** monthly search interest
- **Merges** new 12,544 places with existing 323 Phase 1 places

**Input:** `india_places_clean.csv`
**Output:** `india_crowd_enhanced_v6_FINAL.csv` (12,655 places × 95 cols)

---
> ⚠️ **WARNING: DO NOT RE-RUN FETCH CELLS**
> Popular Times fetch cost ~$19 in API credits.
> All output files are already saved to Google Drive.

In [ ]:
# ── Setup ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, json, os, time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    from dotenv import load_dotenv
    load_dotenv()
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE_API_KEY not set. Add it to the .env file or set as environment variable.')

# Load clean dataset
df = pd.read_csv('/content/drive/MyDrive/india_places_clean.csv')
print(f'✅ Loaded: {df.shape}')

## Cell 1 — Load Place IDs from Checkpoint

In [ ]:
# ─────────────────────────────────────────────────
# CELL 1: Load Place IDs + Merge back into dataset
# ─────────────────────────────────────────────────
import json, os

CHECKPOINT_PATH = '/content/drive/MyDrive/places_raw_checkpoint.json'

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        raw_places = json.load(f)

    df_raw_pids = pd.DataFrame(raw_places)
    print(f'✅ Checkpoint loaded: {len(df_raw_pids):,} places')

    # Build lookup: place_name + city → place_id_google
    df_raw_pids['name_key'] = df_raw_pids['place_name'].str.lower().str.strip()
    df_raw_pids['city_key'] = df_raw_pids['search_city'].str.lower().str.strip()
    pid_lookup = df_raw_pids.drop_duplicates(
        subset=['name_key','city_key']
    ).set_index(['name_key','city_key'])['place_id_google'].to_dict()

    # Match Place IDs
    df['name_key'] = df['place_name'].str.lower().str.strip()
    df['city_key'] = df['city'].str.lower().str.strip()
    df['place_id_google'] = df.apply(
        lambda r: pid_lookup.get((r['name_key'], r['city_key'])), axis=1)
    df = df.drop(columns=['name_key', 'city_key'])

    has_pid = df['place_id_google'].notna().sum()
    print(f'\nWith Place ID   : {has_pid:,} ({has_pid/len(df)*100:.1f}%)')
    print(f'Without Place ID: {df["place_id_google"].isna().sum():,}')

    df.to_csv('/content/drive/MyDrive/india_crowd_forecast_v2_final.csv', index=False)
    print(f'\n💾 Saved with Place IDs!')
else:
    print('❌ Checkpoint not found — run notebook 01 first')

## Cell 2 — Popular Times Fetch
> ⚠️ DO NOT RE-RUN — costs ~$19 in API credits. Output saved: `popular_times_real.json`

In [ ]:
# ─────────────────────────────────────────────────
# CELL 2: Popular Times Fetch
# ⚠️ DO NOT RE-RUN — ~$19 API cost
# Strategy: Only fetch places with 1000+ reviews
# Real data achieved: 9.7% of all places
# ─────────────────────────────────────────────────
!pip install -q git+https://github.com/m-wrzr/populartimes.git python-dotenv
import populartimes

PT_CHECKPOINT = '/content/drive/MyDrive/popular_times_real.json'
DAYS = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']

# Synthetic busyness patterns (fallback for places without real data)
PATTERNS = {
    'Museum'        : dict(o=10,c=18,wkd=58,wke=80,mor=0.50,aft=0.92,eve=0.12),
    'Religious'     : dict(o=5, c=21,wkd=70,wke=90,mor=0.92,aft=0.52,eve=0.90),
    'Heritage'      : dict(o=9, c=18,wkd=55,wke=78,mor=0.62,aft=0.90,eve=0.18),
    'Beach'         : dict(o=6, c=20,wkd=36,wke=84,mor=0.74,aft=0.80,eve=0.92),
    'Park'          : dict(o=6, c=21,wkd=40,wke=76,mor=0.84,aft=0.44,eve=0.74),
    'Wildlife'      : dict(o=6, c=17,wkd=44,wke=80,mor=0.90,aft=0.68,eve=0.08),
    'Hill Station'  : dict(o=8, c=19,wkd=40,wke=90,mor=0.55,aft=0.82,eve=0.55),
    'Nature'        : dict(o=7, c=18,wkd=40,wke=86,mor=0.64,aft=0.82,eve=0.30),
    'Market'        : dict(o=9, c=21,wkd=60,wke=86,mor=0.34,aft=0.74,eve=0.96),
    'Cave'          : dict(o=9, c=17,wkd=50,wke=74,mor=0.54,aft=0.90,eve=0.10),
    'Amusement Park': dict(o=10,c=20,wkd=50,wke=95,mor=0.32,aft=0.82,eve=0.90),
    'Tourist Spot'  : dict(o=9, c=18,wkd=48,wke=76,mor=0.58,aft=0.86,eve=0.26),
}

def make_synthetic(ptype, rating=4.0, seed=42):
    pat = PATTERNS.get(ptype, PATTERNS['Tourist Spot'])
    np.random.seed(seed % 9999)
    mid = (pat['o'] + pat['c']) // 2
    r_b = 1 + (float(rating or 4.0) - 3.5) * 0.08
    result = {}
    for di, day in enumerate(DAYS):
        arr = np.zeros(24)
        is_wke = di >= 5
        base   = pat['wke'] if is_wke else pat['wkd']
        for h in range(24):
            if h < pat['o'] or h >= pat['c']: continue
            tod = (pat['mor'] if h<12 else pat['aft'] if h<18 else pat['eve'])
            arr[h] = min(100, base * tod * max(0.35, 1.0-0.03*abs(h-mid)) * r_b)
        noise  = np.random.normal(0, 3, 24)
        arr    = np.clip(arr + noise*(arr>0), 0, 100)
        result[day] = arr.astype(int).tolist()
    return result

# Load checkpoint
if os.path.exists(PT_CHECKPOINT):
    with open(PT_CHECKPOINT) as f: pt_data = json.load(f)
    done_pids = set(pt_data.keys())
    real_count = sum(1 for v in pt_data.values() if v.get('is_real'))
    print(f'♻️  Checkpoint: {len(done_pids):,} done, {real_count:,} real ({real_count/len(done_pids)*100:.1f}%)')
else:
    pt_data, done_pids = {}, set()

# Filter: 1000+ reviews only
df_fetch = pd.read_csv('/content/drive/MyDrive/india_crowd_forecast_v2_final.csv')
to_fetch = df_fetch[
    (df_fetch['place_id_google'].notna()) &
    (df_fetch['num_reviews'] >= 1000) &
    (~df_fetch['place_id_google'].isin(done_pids))
].sort_values('num_reviews', ascending=False)

print(f'Places to fetch: {len(to_fetch):,}')
print(f'Est. cost      : ~${len(to_fetch)*0.017:.0f}')

# ── Fetch loop ────────────────────────────────────
for i, (_, row) in enumerate(tqdm(to_fetch.iterrows(), total=len(to_fetch))):
    pid    = str(row['place_id_google'])
    ptype  = str(row.get('place_type','Tourist Spot'))
    rating = float(row.get('rating') or 4.0)
    seed   = abs(hash(pid)) % 9999
    pt, is_real = None, False
    try:
        result = populartimes.get_id(GOOGLE_API_KEY, pid)
        if result and result.get('populartimes'):
            pt = {d['name'].lower(): (d['data']+[0]*24)[:24] for d in result['populartimes']}
            is_real = True
    except: pass
    if pt is None:
        pt = make_synthetic(ptype, rating, seed)
    pt_data[pid] = {'place_id': str(row['place_id']), 'place_name': row['place_name'],
                    'is_real': is_real, 'popular_times': pt}
    done_pids.add(pid)
    time.sleep(0.35 if is_real else 0.1)
    if (i+1) % 100 == 0:
        with open(PT_CHECKPOINT, 'w') as f: json.dump(pt_data, f)

with open(PT_CHECKPOINT, 'w') as f: json.dump(pt_data, f)
total_real = sum(1 for v in pt_data.values() if v.get('is_real'))
print(f'\n✅ Done! Total real: {total_real:,} ({total_real/12655*100:.1f}%)')
print(f'💾 Saved → {PT_CHECKPOINT}')

## Cell 3 — Final Merge + Save Enhanced Dataset

In [ ]:
# ─────────────────────────────────────────────────
# CELL 3: Merge Popular Times + Build Final Dataset
# Output: india_crowd_enhanced_v6_FINAL.csv
# ─────────────────────────────────────────────────
import pandas as pd, numpy as np, json

df = pd.read_csv('/content/drive/MyDrive/india_crowd_forecast_v2_final.csv')

with open('/content/drive/MyDrive/popular_times_real.json') as f:
    pt_data = json.load(f)

DAYS = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']

# Merge busyness features
def get_busyness_features(place_id_google, place_id):
    pid_str = str(place_id_google) if pd.notna(place_id_google) else None
    data = pt_data.get(pid_str, {})
    pt   = data.get('popular_times', {})
    is_real = data.get('is_real', False)
    if not pt: return {}
    all_vals = [h for day in DAYS for h in pt.get(day, []) if h > 0]
    wkd_vals = [h for day in DAYS[:5] for h in pt.get(day, []) if h > 0]
    wke_vals = [h for day in DAYS[5:] for h in pt.get(day, []) if h > 0]
    return {
        'busyness_avg':   round(np.mean(all_vals), 1) if all_vals else None,
        'busyness_max':   round(np.max(all_vals), 1) if all_vals else None,
        'weekday_avg':    round(np.mean(wkd_vals), 1) if wkd_vals else None,
        'weekend_avg':    round(np.mean(wke_vals), 1) if wke_vals else None,
        'pt_source':      'google_real' if is_real else 'synthetic',
    }

print('Merging busyness data...')
busyness_rows = [get_busyness_features(row['place_id_google'], row['place_id'])
                  for _, row in df.iterrows()]
df_busy = pd.DataFrame(busyness_rows, index=df.index)
df = pd.concat([df, df_busy], axis=1)

real_count = (df['pt_source'] == 'google_real').sum()
print(f'✅ Merged! Real busyness: {real_count:,} places ({real_count/len(df)*100:.1f}%)')

OUT = '/content/drive/MyDrive/india_crowd_enhanced_v6_FINAL.csv'
df.to_csv(OUT, index=False)
print(f'\n✅ Final dataset: {df.shape}')
print(f'💾 Saved → {OUT}')